# Lasso prediction

In this exercise, I applied Lasso prediction to real data. Specifically, I used wastewater data to predict current Covid cases from frequently updated data. Throughout the pandemic, authorities monitored the presence of Covid in public wastewater to determine whether cases were anticipated to increase or decreased. This exercise attempted to use Lasso to predict near-term case counts from wastewater data.

## Setup

Import the following libraries.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import datetime

## Data preparation and cleaning.

Import and restructure the Biobot wastewater data by county and week. Fill in the blanks below to clean up the data.

In [2]:
url='https://raw.githubusercontent.com/biobotanalytics/covid19-wastewater-data/master/wastewater_by_county.csv'
ww=pd.read_csv(url)

In [3]:
ww2=(ww
     .reset_index()
     .drop(['Unnamed: 0','region', 'index'], axis=1)
     .rename(columns={'sampling_week':'date', 
                      'fipscode':'fips',
                     'effective_concentration_rolling_average':'ww_covid'})
    )

Import the NYT covid county dataset.

In [4]:
url='https://raw.githubusercontent.com/nytimes/covid-19-data/master/us-counties-2022.csv'
covid=pd.read_csv(url)

In [5]:
covid

,date,county,state,fips,cases,deaths
0,2022-01-01,Autauga,Alabama,1001.0,11018,160.0
1,2022-01-01,Baldwin,Alabama,1003.0,39911,593.0
2,2022-01-01,Barbour,Alabama,1005.0,3860,81.0
3,2022-01-01,Bibb,Alabama,1007.0,4533,95.0
4,2022-01-01,Blount,Alabama,1009.0,11256,198.0
...,...,...,...,...,...,...
1015640,2022-11-08,Sweetwater,Wyoming,56037.0,12211,131.0
1015641,2022-11-08,Teton,Wyoming,56039.0,11851,16.0
1015642,2022-11-08,Uinta,Wyoming,56041.0,6210,43.0
1015643,2022-11-08,Washakie,Wyoming,56043.0,2680,47.0


Import 2020 Census data on population by race and ethnicity in each county. I used this to allow the predictions of the model to vary by demography, since Covid did not affect all people and places equally.

In [6]:
filepath='../In-Class-Exercises/data/nhgis0008_csv/nhgis0008_ds248_2020_county.csv'

census1=pd.read_csv(filepath, encoding = "ISO-8859-1")

Import 2010 Census data on age and sex by county. (This information was not yet published for 2020)

In [7]:
filepath='../In-Class-Exercises/data/nhgis0011_csv/nhgis0011_ds176_20105_county.csv'

census2=pd.read_csv(filepath, encoding = "ISO-8859-1")

Create lists of all census variables.

In [8]:
census_features=['Total 2020', 
'Population of one race',
'Population of one race: White alone',
'Population of one race: Black or African American alone',
'Population of one race: American Indian and Alaska Native alone',
'Population of one race: Asian alone',
'Population of one race: Native Hawaiian and Other Pacific Islander alone',
'Population of one race: Some Other Race alone',
'Population of two or more races',
'Hispanic or Latino',
'Total 2010',
'Male: Under 5 years',
'Male: 5 to 9 years',
'Male: 10 to 14 years',
'Male: 15 to 17 years',
'Male: 18 and 19 years',
'Male: 20 years',
'Male: 21 years',
'Male: 22 to 24 years',
'Male: 25 to 29 years',
'Male: 30 to 34 years',
'Male: 35 to 39 years',
'Male: 40 to 44 years',
'Male: 45 to 49 years',
'Male: 50 to 54 years',
'Male: 55 to 59 years',
'Male: 60 and 61 years',
'Male: 62 to 64 years',
'Male: 65 and 66 years',
'Male: 67 to 69 years',
'Male: 70 to 74 years',
'Male: 75 to 79 years',
'Male: 80 to 84 years',
'Male: 85 years and over',
'Female: Under 5 years',
'Female: 5 to 9 years',
'Female: 10 to 14 years',
'Female: 15 to 17 years',
'Female: 18 and 19 years',
'Female: 20 years',
'Female: 21 years',
'Female: 22 to 24 years',
'Female: 25 to 29 years',
'Female: 30 to 34 years',
'Female: 35 to 39 years',
'Female: 40 to 44 years',
'Female: 45 to 49 years',
'Female: 50 to 54 years',
'Female: 55 to 59 years',
'Female: 60 and 61 years',
'Female: 62 to 64 years',
'Female: 65 and 66 years',
'Female: 67 to 69 years',
'Female: 70 to 74 years',
'Female: 75 to 79 years',
'Female: 80 to 84 years',
'Female: 85 years and over']

In [9]:
census_vars=['STATE_x' ,'STATEA_x', 'COUNTY_x','COUNTYA_x']+census_features

Create a dict that maps from the data file codes to the actual meanings I want to use as labels.

In [10]:
census_relabel_dict=({    'U7B001':      'Total 2020',
    'U7B002':      'Population of one race',
    'U7B003':      'Population of one race: White alone',
    'U7B004':      'Population of one race: Black or African American alone',
    'U7B005':      'Population of one race: American Indian and Alaska Native alone',
    'U7B006':      'Population of one race: Asian alone',
    'U7B007':      'Population of one race: Native Hawaiian and Other Pacific Islander alone',
    'U7B008':      'Population of one race: Some Other Race alone',
    'U7B009':      'Population of two or more races',
    'U7C002':      'Hispanic or Latino',
    'U7C003':      'Not Hispanic or Latino',
    'JLZE001':     'Total 2010',
    'JLZE002':     'Male',
    'JLZE003':     'Male: Under 5 years',
    'JLZE004':     'Male: 5 to 9 years',
    'JLZE005':     'Male: 10 to 14 years',
    'JLZE006':     'Male: 15 to 17 years',
    'JLZE007':     'Male: 18 and 19 years',
    'JLZE008':     'Male: 20 years',
    'JLZE009':     'Male: 21 years',
    'JLZE010':     'Male: 22 to 24 years',
    'JLZE011':     'Male: 25 to 29 years',
    'JLZE012':     'Male: 30 to 34 years',
    'JLZE013':     'Male: 35 to 39 years',
    'JLZE014':     'Male: 40 to 44 years',
    'JLZE015':     'Male: 45 to 49 years',
    'JLZE016':     'Male: 50 to 54 years',
    'JLZE017':     'Male: 55 to 59 years',
    'JLZE018':     'Male: 60 and 61 years',
    'JLZE019':     'Male: 62 to 64 years',
    'JLZE020':     'Male: 65 and 66 years',
    'JLZE021':     'Male: 67 to 69 years',
    'JLZE022':     'Male: 70 to 74 years',
    'JLZE023':     'Male: 75 to 79 years',
    'JLZE024':     'Male: 80 to 84 years',
    'JLZE025':     'Male: 85 years and over',
    'JLZE026':     'Female',
    'JLZE027':     'Female: Under 5 years',
    'JLZE028':     'Female: 5 to 9 years',
    'JLZE029':     'Female: 10 to 14 years',
    'JLZE030':     'Female: 15 to 17 years',
    'JLZE031':     'Female: 18 and 19 years',
    'JLZE032':     'Female: 20 years',
    'JLZE033':     'Female: 21 years',
    'JLZE034':     'Female: 22 to 24 years',
    'JLZE035':     'Female: 25 to 29 years',
    'JLZE036':     'Female: 30 to 34 years',
    'JLZE037':     'Female: 35 to 39 years',
    'JLZE038':     'Female: 40 to 44 years',
    'JLZE039':     'Female: 45 to 49 years',
    'JLZE040':     'Female: 50 to 54 years',
    'JLZE041':     'Female: 55 to 59 years',
    'JLZE042':     'Female: 60 and 61 years',
    'JLZE043':     'Female: 62 to 64 years',
    'JLZE044':     'Female: 65 and 66 years',
    'JLZE045':     'Female: 67 to 69 years',
    'JLZE046':     'Female: 70 to 74 years',
    'JLZE047':     'Female: 75 to 79 years',
    'JLZE048':     'Female: 80 to 84 years',
    'JLZE049':     'Female: 85 years and over'})

Merge the census1 and census2 datasets to get a single Census dataset. Fill in the blanks to apply the relabeling dictionary above to create coherent column names.

In [11]:
census=(census1
        .merge(census2, on='GISJOIN')
        .rename(columns=census_relabel_dict)
        .loc[:,census_vars]
       )

# Create a fips variable for the census data
census['fips']=census['STATEA_x']*1000+census['COUNTYA_x']


Merge the wastewater, census, and covid case data into a single dataframe. Set the `how` parameter such that the final data frame has one observation per county per week in each row, without missing wastewater data. Recall that the census data is from a single census and does not change over `date`.

In [12]:
df=(ww2
    .merge(covid, on=['date','fips'],how='left')
    .merge(census, on=['fips'], how='left')
    .sort_values(['fips','date'])
   )

Take a look at our merged data.

In [13]:
df.head()

,date,ww_covid,state_x,name,fips,county,state_y,cases,deaths,STATE_x,...,Female: 50 to 54 years,Female: 55 to 59 years,Female: 60 and 61 years,Female: 62 to 64 years,Female: 65 and 66 years,Female: 67 to 69 years,Female: 70 to 74 years,Female: 75 to 79 years,Female: 80 to 84 years,Female: 85 years and over
6746,2022-07-20,1193.752341,AL,"Colbert County, AL",1033,Colbert,Alabama,17641.0,264.0,Alabama,...,2092.0,1951.0,805.0,902.0,678.0,894.0,1209.0,1041.0,886.0,708.0
6941,2022-07-27,1406.164557,AL,"Colbert County, AL",1033,Colbert,Alabama,17852.0,264.0,Alabama,...,2092.0,1951.0,805.0,902.0,678.0,894.0,1209.0,1041.0,886.0,708.0
7140,2022-08-03,1188.332168,AL,"Colbert County, AL",1033,Colbert,Alabama,17945.0,264.0,Alabama,...,2092.0,1951.0,805.0,902.0,678.0,894.0,1209.0,1041.0,886.0,708.0
7347,2022-08-10,1288.395508,AL,"Colbert County, AL",1033,Colbert,Alabama,18128.0,265.0,Alabama,...,2092.0,1951.0,805.0,902.0,678.0,894.0,1209.0,1041.0,886.0,708.0
7558,2022-08-17,1016.000715,AL,"Colbert County, AL",1033,Colbert,Alabama,18356.0,266.0,Alabama,...,2092.0,1951.0,805.0,902.0,678.0,894.0,1209.0,1041.0,886.0,708.0


Fillin the gaps below to create a value, `end_date`, that I will use to define where the data is in the future vs. the past and present.

In [14]:
today=datetime.date.today()

end_date=str(today)

print(end_date)

2022-11-10


Create an multi-index by fips and date, with date in datetime format. The fips variable is a numeric ID for each county.

In [15]:
df['date']=pd.to_datetime(df['date'])


In [16]:
df=df.set_index(['fips','date'])

Create additional dataframe rows for weeks into the future.

In [17]:
# Create a dataframe with dates for the weeks following the end of the current wastewater observations.
# Predictions for future covid are going to go here!

df_future=(df
           .reset_index()
           .groupby('fips')
           .tail(7)
          )

# Create dates shifted into the future. 
df_future['date']=df_future['date']+datetime.timedelta(weeks=7)

df_future=df_future[df_future['date']>end_date].set_index(['fips','date'])

# Shift will create false values for variables carreid over from the shifted
# time period. Replace the time-varying data we don't know yet with missing values
time_varying_variables=['ww_covid','cases','deaths']
for var in time_varying_variables:
    df_future[var]=np.nan
    
# Put these future dates back into the main dataframe
df=(pd.concat([df, df_future])
    .reset_index()
    .sort_values(['fips','date'])
    .set_index(['fips','date']))

Create variables for previous weeks' wastewater data.

In [18]:
for i in range(1,8):
    varname = 'ww_covid_lag'+str(i)
    df[varname]=df.groupby(level='fips')['ww_covid'].shift(i)

Create new variables for the weekly change in cases and deaths.

In [19]:
df['new_cases']=df['cases']-df.groupby(level='fips')['cases'].shift(1)
df['new_deaths']=df['deaths']-df.groupby(level='fips')['deaths'].shift(1)

Discard rows where wastewater data is missing

In [20]:
# Sci-kit learn gets grumpy if variables are incompletely observed, 
# so we need to drop every row where the last lag is missing.

df_noindex=df
df_nomissing=(df
              .loc[
                  (df['ww_covid_lag7'].notnull()) 
                  & (df['ww_covid_lag6'].notnull())
                  & (df['ww_covid_lag5'].notnull())
                  & (df['ww_covid_lag4'].notnull())
                  & (df['ww_covid_lag3'].notnull())
                  & (df['ww_covid_lag2'].notnull())
                  & (df['ww_covid_lag1'].notnull())
                  ]
                )

Add a list of the lagged case counts from previous weeks to the set of predictor variables

In [21]:
all_features = census_features+['ww_covid_lag1','ww_covid_lag2','ww_covid_lag3','ww_covid_lag4','ww_covid_lag5','ww_covid_lag6','ww_covid_lag7'] 

Can't use the `.dropna()` command, because we've created weeks for future periods we hope to forecast.

Instead, fill in the blanks so that the loop below drops observations where we don't observe all the features.

In [22]:
for i in all_features:
    df_nomissing=df_nomissing.loc[df_nomissing[i].notna()]

## Prepare the dataset for lasso

Use `PolynomialFeatures` to create squares and interactions of all of our features of interest: the census data and time-varying lagged wastewater values.



In [ ]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(2)

poly_array=poly.fit_transform(df_nomissing[all_features])

poly_features=poly.get_feature_names_out(df_nomissing[all_features].columns)

poly_features_df=pd.DataFrame(poly_array,columns=poly_features)

df_outcomes=df_nomissing[['cases','deaths','new_cases','new_deaths']].reset_index()

df_poly=(df_outcomes
         .join(poly_features_df, how='outer')
        )

Split `df_poly` into two data frames, one where all data is historical (older than today) and one with future dates we are trying to forecast.



In [ ]:
training_base=(df_poly
               .loc[(df_poly['date']<=end_date) 
                    & ~(df_poly['new_cases'].isnull())
                ] )
forecasting_target=df_poly.loc[(df_poly['date']>end_date)]

Use `test_train_split` to divide the past/present data into training and testing subsets.

In [ ]:
# Split into testing and training data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = (train_test_split(training_base[poly_features], 
                                                     training_base['new_cases'], 
                                                     test_size=0.3, 
                                                     random_state=0))




Use `StandardScaler` to convert all features so that the training data set is mean 0, sd 1. Apply that rescaling to teh testing and target data as well. For testing purposes, create a rescaled dataset that contains the testing and training data together, rescaled.



In [ ]:
# Put the data on a standard scale, normalized using the training data
from sklearn.preprocessing import StandardScaler

# Define the rescaling in terms of the numerical regressors in the training data
scaler = StandardScaler().fit(X_train) 

# Rescale the training data
X_train = scaler.transform(X_train)

# Use the same transformation on the testing data
X_test = scaler.transform(X_test)

# And on the total dataset
X_pooled=scaler.transform(training_base[poly_features])
y_pooled=training_base['new_cases']

# And on the forecasting target
X_target= scaler.transform(forecasting_target[poly_features])

## Estimate Lasso models and forecast Covid cases

Use the split and rescaled data to fit a Lasso model with alpha=100. Compare goodness of fit

In [ ]:
# Try a Lasso with specific alpha
from sklearn.linear_model import Lasso
lasso_reg = Lasso(alpha=1_00)

# Fit the model using the training data
lasso_reg.fit(X_train, y_train)

# Create predicted outcomes for the training data
yhat_train=lasso_reg.predict(X_train)
print('R-squared on training data: '+str(round(sklearn.metrics.r2_score(y_train, yhat_train)*100,1))+"%")

# Create predicted outcomes for the testing data
yhat_test=lasso_reg.predict(X_test)
print('R-squared on testing data: '+str(round(sklearn.metrics.r2_score(y_test, yhat_test)*100,1))+"%")

Loop over a variety of alpha values and save the goodness-of-fit in a dataframe. 

In [ ]:
# Fit the model and save R-squared for many different values of alpha

start=10
stop=500
step=10

lasso_results=np.empty([(stop-start)//step,3])

for myalpha in range(start, stop, step):
    index=(myalpha-start)//step
    
    lasso_results[index,0]=myalpha
    
    lasso_reg = Lasso(alpha=myalpha, tol=0.1)
    lasso_reg.fit(X_train, y_train)
    
    yhat_train=lasso_reg.predict(X_train)
    yhat_test=lasso_reg.predict(X_test)
    
    lasso_results[index,1]=sklearn.metrics.r2_score(y_train, yhat_train)*100 
    lasso_results[index,2]=sklearn.metrics.r2_score(y_test, yhat_test)*100
    
lasso_df=pd.DataFrame(lasso_results,columns=['alpha','Training R2', 'Test R2'])

Plot the R-squared data for testing and training data. Observe whic alpha gives the best predictions out of sample.

In [ ]:
# Plot R-squared in and out of sample as alpha changes

g=sns.lineplot(data=lasso_df, x='alpha', y='Training R2', label='Training Data')
g=sns.lineplot(data=lasso_df, x='alpha', y='Test R2', label='Test Data', linestyle='dashed', color='g')

plt.xlabel('Alpha')
plt.ylabel('R2')

plt.legend()

plt.show()

Repeat the fit, but now use `LassoCV` to search for the best alpha using cross-validation over the `pooled` dataset.

In [ ]:
# rom sklearn.linear_model import LassoCV
lassocv_reg = LassoCV(tol=0.01, max_iter=2000, random_state=0)

lassocv_reg.fit(X_pooled, y_pooled)

print('Alpha chosen by cross-validation: '+str(round(lassocv_reg.alpha_,3)))

Add a vertical line to our plot comparing the CV alpha to testing and training R-squareds from our loop.

In [ ]:
# How does the CV alpha compare to our looped results? Add a vertical line to the plot
g=sns.lineplot(data=lasso_df, x='alpha', y='Training R2', label='Training Data')
g=sns.lineplot(data=lasso_df, x='alpha', y='Test R2', label='Test Data', linestyle='dashed', color='g')


plt.xlabel('Alpha')
plt.ylabel('R2')

plt.legend()

plt.axvline(x=lassocv_reg.alpha_, color='red', linestyle='dotted')
plt.show()

*As of this writing, the cross-validated lasso and manually tuned lasso do not agree at all! This can have a few different causes. For one, the best hyperparameter might depend a lot on the random split of the data into testing and training sets, which would be re-randomized with each fold of the cross-validation method. On the other hand, the LassoCV search might be getting stuck near a local maximum, with its coordinate search failing to see a higher max to the right. Which is the problem would require more substantial tinkering and debugging than I want to bother with today.*

It looks like an alpha of 300 is about the ideal from the manually tuned lasso. Use it to fit my model of choice.

In [ ]:
# Try a Lasso with specific alpha
from sklearn.linear_model import Lasso
lasso_reg_final = Lasso(alpha=300)

# Fit the model using the training data
lasso_reg_final.fit(X_train, y_train)

# Create predicted outcomes for the training data
yhat_train=lasso_reg_final.predict(X_train)
print('R-squared on training data: '+str(round(sklearn.metrics.r2_score(y_train, yhat_train)*100,1))+"%")

# Create predicted outcomes for the testing data
yhat_test=lasso_reg_final.predict(X_test)
print('R-squared on testing data: '+str(round(sklearn.metrics.r2_score(y_test, yhat_test)*100,1))+"%")

Use the `lasso_reg_final` model to predict new case counts for future weeks.

In [ ]:
# Create lasso predictions for recent periods where we don't know the tax revenue yet
lasso_predictions=lasso_reg_final.predict(X_target)

# Define a variable called "end index" that contains the data (index) values for the period I'm trying to predict. This will make merging easier.
end_index=forecasting_target.index 

# Put the lasso predictions, which come out as an array, into a dataframe using the index values
df_lasso_pred=pd.DataFrame(lasso_predictions, index=end_index, columns=['Lasso Prediction'])

# Join with existing dataframes
df_predictions=forecasting_target.join(df_lasso_pred)
df_final=pd.concat([df_predictions,training_base])

For graphing, look at how actual new case counts in Boston look for my prediction relative to history. Export a subset of `df_final` for the FIPS code 25025 (Suffolk County, Massachusetts). This is the county with the longest history of wastewater data, so I'll be able to see the predictions in full context.

In [ ]:
df_boston=(df_final[df_final['fips']==25025]
           .sort_values('date')
          )

Make a Seaborn line plot of the actual cases in Suffolk County in blue, and prediction of our LASSO model for next week using a red dot. 

In [ ]:
sns.lineplot(data=df_boston, y='new_cases', x='date', color='blue', label='Historic cases')
sns.scatterplot(data=df_boston, y='Lasso Prediction', x='date', color='red', label='Lasso Prediction')

plt.show()

*As of this writing, the Seaborn model is expecting case counts that look a little bit too high. This arguably shows the limits of predictive models -- they only work so long as the underlying data generating process is stable. In the case of covid, we've used a period when publicly reported PCR testing was common and vaccination not as widespread to make our predictions. Now that we're in a period of at-home antigen testing (which doesn't show up in case counts) and high collective immunity, yesterday's cases don't predict big observed case spikes today.*

*Then again... we are heading into the cold weather. If Boston-area cases spike a bit in the next few weeks, then that's a strong point in favor of our model.*

**Update:** *Having re-run this code on 11/3/2022... the modest spike that the lasso model was predicting has happened. Fortunately, as of THIS writing, further increases are not predicted.*